# Prompt 实验三：冲突上下文

给模型两份版本不同、口径冲突的指标文档，比较三种方案：

1. `no_priority`：新旧文档都进入上下文，不提供优先级规则。
2. `priority_rule`：新旧文档都进入上下文，明确按生效日期和版本选择。
3. `latest_only`：程序层只保留最新版文档，Prompt 不处理冲突。

每组指标生成“旧文档在前”和“新文档在前”两个样本，用来观察模型是否机械选择最后一份文档。

> 全量实验会发起 60 次 API 请求。先运行 1 条样本的冒烟实验，共 3 次请求。

In [8]:
import getpass
import json
import os
import time
from pathlib import Path
from typing import Literal

import requests
from pydantic import BaseModel, Field, ValidationError

if not os.getenv("Z_API_KEY"):
    os.environ["Z_API_KEY"] = getpass.getpass("请输入智谱 API Key：")

print("Z_API_KEY 已设置：", bool(os.getenv("Z_API_KEY")))

Z_API_KEY 已设置： True


In [9]:
API_URL = "https://open.bigmodel.cn/api/paas/v4/chat/completions"
MODEL = "glm-4.5-air"
TIMEOUT_SECONDS = 60
VARIANTS = ("no_priority", "priority_rule", "latest_only")


class MetricAnswer(BaseModel):
    metric_id: str = Field(min_length=1, max_length=40)
    definition_id: str = Field(min_length=1, max_length=50)
    source_id: str = Field(min_length=1, max_length=60)
    version: str = Field(min_length=1, max_length=20)
    effective_date: str = Field(min_length=10, max_length=10)
    reason: str = Field(min_length=1, max_length=60)

## 1. 建立冲突指标文档

每组文档都有来源、版本和生效日期，但定义内容互相冲突。`is_latest` 模拟检索系统中的版本元数据。

In [10]:
METRIC_PAIRS = [
    {
        "metric_id": "conversion_rate",
        "metric_name": "转化率",
        "old": {
            "source_id": "metric_conversion_v1",
            "version": "1.0",
            "effective_date": "2024-01-01",
            "definition_id": "conversion_orders_visits",
            "definition": "下单次数除以访问次数",
        },
        "latest": {
            "source_id": "metric_conversion_v2",
            "version": "2.0",
            "effective_date": "2025-01-01",
            "definition_id": "conversion_paid_unique",
            "definition": "支付用户数除以独立访客数",
        },
    },
    {
        "metric_id": "active_user",
        "metric_name": "活跃用户",
        "old": {
            "source_id": "metric_active_user_v1",
            "version": "1.3",
            "effective_date": "2024-02-01",
            "definition_id": "active_login_30d",
            "definition": "最近 30 天内登录过的用户",
        },
        "latest": {
            "source_id": "metric_active_user_v2",
            "version": "2.1",
            "effective_date": "2025-02-15",
            "definition_id": "active_action_7d",
            "definition": "最近 7 天内完成过核心行为的用户",
        },
    },
    {
        "metric_id": "revenue",
        "metric_name": "营业收入",
        "old": {
            "source_id": "metric_revenue_v1",
            "version": "1.0",
            "effective_date": "2024-03-01",
            "definition_id": "revenue_gross_tax",
            "definition": "含税订单金额，不扣除退款",
        },
        "latest": {
            "source_id": "metric_revenue_v2",
            "version": "2.0",
            "effective_date": "2025-03-01",
            "definition_id": "revenue_net_refund",
            "definition": "不含税支付金额，扣除已完成退款",
        },
    },
    {
        "metric_id": "churn_rate",
        "metric_name": "流失率",
        "old": {
            "source_id": "metric_churn_v1",
            "version": "1.2",
            "effective_date": "2024-04-01",
            "definition_id": "churn_end_base",
            "definition": "本期流失用户数除以期末用户数",
        },
        "latest": {
            "source_id": "metric_churn_v2",
            "version": "2.0",
            "effective_date": "2025-04-01",
            "definition_id": "churn_start_base",
            "definition": "本期流失用户数除以期初用户数",
        },
    },
    {
        "metric_id": "response_time",
        "metric_name": "响应时长",
        "old": {
            "source_id": "metric_response_time_v1",
            "version": "1.0",
            "effective_date": "2024-05-01",
            "definition_id": "response_average",
            "definition": "全部请求响应时长的算术平均值",
        },
        "latest": {
            "source_id": "metric_response_time_v2",
            "version": "2.2",
            "effective_date": "2025-05-01",
            "definition_id": "response_p95",
            "definition": "请求响应时长的 P95 值",
        },
    },
    {
        "metric_id": "available_inventory",
        "metric_name": "可用库存",
        "old": {
            "source_id": "metric_inventory_v1",
            "version": "1.1",
            "effective_date": "2024-06-01",
            "definition_id": "inventory_on_hand",
            "definition": "仓库当前在手数量",
        },
        "latest": {
            "source_id": "metric_inventory_v2",
            "version": "2.0",
            "effective_date": "2025-06-01",
            "definition_id": "inventory_minus_reserved",
            "definition": "在手数量减去已预占数量",
        },
    },
    {
        "metric_id": "new_user",
        "metric_name": "新增用户",
        "old": {
            "source_id": "metric_new_user_v1",
            "version": "1.4",
            "effective_date": "2024-07-01",
            "definition_id": "new_user_registered",
            "definition": "统计期内完成注册的用户",
        },
        "latest": {
            "source_id": "metric_new_user_v2",
            "version": "2.0",
            "effective_date": "2025-07-01",
            "definition_id": "new_user_activated",
            "definition": "统计期内首次完成激活的用户",
        },
    },
    {
        "metric_id": "refund_rate",
        "metric_name": "退款率",
        "old": {
            "source_id": "metric_refund_v1",
            "version": "1.0",
            "effective_date": "2024-08-01",
            "definition_id": "refund_requested_created",
            "definition": "退款申请数除以创建订单数",
        },
        "latest": {
            "source_id": "metric_refund_v2",
            "version": "2.3",
            "effective_date": "2025-08-01",
            "definition_id": "refund_completed_paid",
            "definition": "完成退款数除以支付订单数",
        },
    },
    {
        "metric_id": "day7_retention",
        "metric_name": "7 日留存率",
        "old": {
            "source_id": "metric_retention_v1",
            "version": "1.2",
            "effective_date": "2024-09-01",
            "definition_id": "retention_login_day7",
            "definition": "第 7 天登录用户数除以新增用户数",
        },
        "latest": {
            "source_id": "metric_retention_v2",
            "version": "2.0",
            "effective_date": "2025-09-01",
            "definition_id": "retention_action_day7",
            "definition": "第 7 天完成核心行为用户数除以激活用户数",
        },
    },
    {
        "metric_id": "order_count",
        "metric_name": "订单量",
        "old": {
            "source_id": "metric_order_count_v1",
            "version": "1.0",
            "effective_date": "2024-10-01",
            "definition_id": "orders_created",
            "definition": "统计期内创建的订单数量",
        },
        "latest": {
            "source_id": "metric_order_count_v2",
            "version": "2.0",
            "effective_date": "2025-10-01",
            "definition_id": "orders_paid",
            "definition": "统计期内支付成功的订单数量",
        },
    },
]

for pair in METRIC_PAIRS:
    pair["old"]["metric_id"] = pair["metric_id"]
    pair["old"]["is_latest"] = False
    pair["latest"]["metric_id"] = pair["metric_id"]
    pair["latest"]["is_latest"] = True

print("冲突指标组数：", len(METRIC_PAIRS))

冲突指标组数： 10


## 2. 生成 20 条固定测试数据

同一指标生成两种文档顺序。预期答案在调用模型之前由 `latest` 元数据确定，不能根据模型输出修改。

In [11]:
CASES = []

for index, pair in enumerate(METRIC_PAIRS, start=1):
    for order_name, documents in (
        ("old_first", [pair["old"], pair["latest"]]),
        ("new_first", [pair["latest"], pair["old"]]),
    ):
        latest = pair["latest"]
        CASES.append({
            "id": f"c{index:02d}_{order_name}",
            "question": f"请给出公司当前采用的“{pair['metric_name']}”指标口径。",
            "metric_id": pair["metric_id"],
            "documents": documents,
            "expected": {
                "metric_id": pair["metric_id"],
                "definition_id": latest["definition_id"],
                "source_id": latest["source_id"],
                "version": latest["version"],
                "effective_date": latest["effective_date"],
            },
            "tags": [order_name, pair["metric_id"]],
        })

assert len(CASES) == 20
print("测试样本数：", len(CASES))
print("第一个样本：", CASES[0]["id"], CASES[0]["question"])
print("第二个样本的文档顺序：", [doc["source_id"] for doc in CASES[1]["documents"]])

测试样本数： 20
第一个样本： c01_old_first 请给出公司当前采用的“转化率”指标口径。
第二个样本的文档顺序： ['metric_conversion_v2', 'metric_conversion_v1']


## 3. 构造三种上下文方案

`no_priority` 和 `latest_only` 使用相同基础指令；后者只在程序层过滤上下文。`priority_rule` 保留两份文档，并增加明确的冲突处理规则。

In [12]:
ANSWER_SCHEMA = {
    "type": "object",
    "properties": {
        "metric_id": {"type": "string"},
        "definition_id": {"type": "string"},
        "source_id": {"type": "string"},
        "version": {"type": "string"},
        "effective_date": {"type": "string"},
        "reason": {"type": "string", "minLength": 1, "maxLength": 60},
    },
    "required": [
        "metric_id",
        "definition_id",
        "source_id",
        "version",
        "effective_date",
        "reason",
    ],
    "additionalProperties": False,
}

BASE_INSTRUCTION = f"""你是公司指标口径查询助手。
任务：只根据提供的指标文档，选择公司当前采用的指标口径及其来源。
边界：不要使用外部知识，不要编造文档中不存在的来源、版本、日期或定义。
输出：只返回严格符合下面 JSON Schema 的 JSON，不要输出 Markdown 或额外文字。

JSON Schema：
{json.dumps(ANSWER_SCHEMA, ensure_ascii=False, indent=2)}"""

PRIORITY_RULE = """冲突处理规则：
1. 优先选择 effective_date 较新的文档。
2. effective_date 相同时，选择 version 较高的文档。
3. 文档在上下文中的先后顺序不代表优先级。"""


def select_context_documents(case: dict, variant: str) -> list[dict]:
    if variant == "latest_only":
        return [doc for doc in case["documents"] if doc["is_latest"]]
    if variant in ("no_priority", "priority_rule"):
        return case["documents"]
    raise ValueError(f"未知版本：{variant}")


def build_messages(case: dict, variant: str) -> tuple[list[dict[str, str]], list[dict]]:
    documents = select_context_documents(case, variant)
    instruction = BASE_INSTRUCTION
    if variant == "priority_rule":
        instruction += "\n\n" + PRIORITY_RULE

    user_content = (
        "指标文档：\n"
        + json.dumps(documents, ensure_ascii=False, indent=2)
        + "\n\n用户问题：\n"
        + case["question"]
    )

    messages = [
        {"role": "system", "content": instruction},
        {"role": "user", "content": user_content},
    ]
    return messages, documents


for variant in VARIANTS:
    _, docs = build_messages(CASES[1], variant)
    print(variant, "上下文文档：", [doc["source_id"] for doc in docs])

no_priority 上下文文档： ['metric_conversion_v2', 'metric_conversion_v1']
priority_rule 上下文文档： ['metric_conversion_v2', 'metric_conversion_v1']
latest_only 上下文文档： ['metric_conversion_v2']


## 4. 调用智谱 API

模型、采样参数、JSON 输出模式在三个版本中完全相同。

In [13]:
def call_glm(messages: list[dict[str, str]]) -> dict:
    api_key = os.getenv("Z_API_KEY")
    if not api_key:
        raise RuntimeError("没有获取到 Z_API_KEY")

    payload = {
        "model": MODEL,
        "messages": messages,
        "thinking": {"type": "disabled"},
        "temperature": 0.1,
        "max_tokens": 192,
        "stream": False,
        "response_format": {"type": "json_object"},
    }

    started = time.perf_counter()
    response = requests.post(
        API_URL,
        headers={
            "Authorization": f"Bearer {api_key}",
            "Content-Type": "application/json",
        },
        json=payload,
        timeout=TIMEOUT_SECONDS,
    )
    elapsed = time.perf_counter() - started

    if not response.ok:
        raise RuntimeError(f"HTTP {response.status_code}: {response.text[:500]}")

    data = response.json()
    return {
        "content": data["choices"][0]["message"]["content"],
        "elapsed_seconds": round(elapsed, 3),
        "usage": data.get("usage", {}),
        "request_id": data.get("id") or data.get("request_id"),
    }

## 5. 解析和评测

`fully_correct` 要求定义、来源、版本和日期全部正确。另行记录是否选择旧文档、是否选择最后一份文档，以及来源是否真实存在于上下文。

In [14]:
def parse_output(raw: str) -> MetricAnswer:
    return MetricAnswer.model_validate_json(raw)


def evaluate_variant(case: dict, variant: str) -> dict:
    messages, context_documents = build_messages(case, variant)
    context_source_ids = [doc["source_id"] for doc in context_documents]
    expected = case["expected"]

    record = {
        "id": case["id"],
        "question": case["question"],
        "metric_id": case["metric_id"],
        "tags": case["tags"],
        "variant": variant,
        "context_source_ids": context_source_ids,
        "last_source_id": context_source_ids[-1],
        "expected": expected,
        "api_ok": False,
        "format_ok": False,
        "predicted": None,
        "definition_correct": False,
        "source_correct": False,
        "fully_correct": False,
        "grounded_source": False,
        "selected_old_source": False,
        "selected_last_document": False,
        "raw": None,
        "elapsed_seconds": None,
        "usage": {},
        "request_id": None,
        "error": None,
    }

    try:
        response = call_glm(messages)
        record.update(
            api_ok=True,
            raw=response["content"],
            elapsed_seconds=response["elapsed_seconds"],
            usage=response["usage"],
            request_id=response["request_id"],
        )

        parsed = parse_output(response["content"])
        predicted = parsed.model_dump()
        definition_correct = parsed.definition_id == expected["definition_id"]
        source_correct = parsed.source_id == expected["source_id"]
        fully_correct = all(
            predicted[field] == expected[field]
            for field in ("metric_id", "definition_id", "source_id", "version", "effective_date")
        )

        record.update(
            format_ok=True,
            predicted=predicted,
            definition_correct=definition_correct,
            source_correct=source_correct,
            fully_correct=fully_correct,
            grounded_source=parsed.source_id in context_source_ids,
            selected_old_source=parsed.source_id != expected["source_id"] and parsed.source_id in context_source_ids,
            selected_last_document=parsed.source_id == context_source_ids[-1],
        )
    except (requests.RequestException, RuntimeError, ValueError, ValidationError) as exc:
        record["error"] = f"{type(exc).__name__}: {exc}"

    return record


def run_experiment(cases: list[dict], delay_seconds: float = 0.3) -> list[dict]:
    records = []
    for index, case in enumerate(cases, start=1):
        for variant in VARIANTS:
            print(f"[{index}/{len(cases)}] {case['id']} - {variant}")
            records.append(evaluate_variant(case, variant))
            time.sleep(delay_seconds)
    return records

## 6. 冒烟实验（共 3 次请求）

选择 `new_first` 样本：最新版在前、旧版在后。无优先级版本如果机械选择最后一份文档，就会选错。

In [15]:
smoke_results = run_experiment([CASES[1]])

for row in smoke_results:
    print()
    print(row["variant"], "上下文：", row["context_source_ids"])
    print("原始输出：", row["raw"])
    print("完全正确：", row["fully_correct"], "选择最后文档：", row["selected_last_document"])
    print("错误：", row["error"])

[1/1] c01_new_first - no_priority
[1/1] c01_new_first - priority_rule
[1/1] c01_new_first - latest_only

no_priority 上下文： ['metric_conversion_v2', 'metric_conversion_v1']
原始输出： {"metric_id":"conversion_rate","definition_id":"conversion_paid_unique","source_id":"metric_conversion_v2","version":"2.0","effective_date":"2025-01-01","reason":"当前最新版本"}
完全正确： True 选择最后文档： False
错误： None

priority_rule 上下文： ['metric_conversion_v2', 'metric_conversion_v1']
原始输出： {"metric_id":"conversion_rate","definition_id":"conversion_paid_unique","source_id":"metric_conversion_v2","version":"2.0","effective_date":"2025-01-01","reason":"最新版本定义"}
完全正确： True 选择最后文档： False
错误： None

latest_only 上下文： ['metric_conversion_v2']
原始输出： {"metric_id":"conversion_rate","definition_id":"conversion_paid_unique","source_id":"metric_conversion_v2","version":"2.0","effective_date":"2025-01-01","reason":"最新版本指标"}
完全正确： True 选择最后文档： True
错误： None


## 7. 汇总指标

除总体准确率外，还要看旧版选择率、来源可追溯率和最后文档选择率。

In [16]:
def summarize(records: list[dict]) -> list[dict]:
    summary = []

    for variant in VARIANTS:
        rows = [row for row in records if row["variant"] == variant]
        api_rows = [row for row in rows if row["api_ok"]]
        parsed_rows = [row for row in rows if row["format_ok"]]
        count = len(rows)
        prompt_tokens = sum(row["usage"].get("prompt_tokens", 0) for row in rows)
        completion_tokens = sum(row["usage"].get("completion_tokens", 0) for row in rows)

        summary.append({
            "variant": variant,
            "samples": count,
            "api_success_rate": round(sum(row["api_ok"] for row in rows) / count, 3) if count else 0,
            "format_pass_rate": round(sum(row["format_ok"] for row in rows) / count, 3) if count else 0,
            "definition_accuracy": round(sum(row["definition_correct"] for row in rows) / count, 3) if count else 0,
            "source_accuracy": round(sum(row["source_correct"] for row in rows) / count, 3) if count else 0,
            "full_accuracy": round(sum(row["fully_correct"] for row in rows) / count, 3) if count else 0,
            "grounded_source_rate": round(sum(row["grounded_source"] for row in rows) / len(parsed_rows), 3) if parsed_rows else 0,
            "old_source_rate": round(sum(row["selected_old_source"] for row in rows) / len(parsed_rows), 3) if parsed_rows else 0,
            "last_document_rate": round(sum(row["selected_last_document"] for row in rows) / len(parsed_rows), 3) if parsed_rows else 0,
            "avg_latency_seconds": (
                round(sum(row["elapsed_seconds"] for row in api_rows) / len(api_rows), 3)
                if api_rows else None
            ),
            "avg_prompt_tokens": round(prompt_tokens / count, 1) if count else 0,
            "avg_completion_tokens": round(completion_tokens / count, 1) if count else 0,
            "total_tokens": prompt_tokens + completion_tokens,
        })

    return summary


def summarize_by_order(records: list[dict]) -> list[dict]:
    order_summary = []
    for variant in VARIANTS:
        for order_name in ("old_first", "new_first"):
            rows = [
                row for row in records
                if row["variant"] == variant and order_name in row["tags"]
            ]
            order_summary.append({
                "variant": variant,
                "document_order": order_name,
                "samples": len(rows),
                "full_accuracy": round(sum(row["fully_correct"] for row in rows) / len(rows), 3) if rows else 0,
                "selected_last_document_rate": round(sum(row["selected_last_document"] for row in rows) / len(rows), 3) if rows else 0,
            })
    return order_summary

## 8. 全量运行并立即保存（共 60 次请求）

确认冒烟实验无鉴权、接口或解析错误后，再运行下面的单元格。

In [17]:
results = run_experiment(CASES)
summary = summarize(results)
order_summary = summarize_by_order(results)

for item in summary:
    print(item)

output_dir = Path.cwd()
if output_dir.name != "02-LLM与Prompt" and (output_dir / "02-LLM与Prompt").is_dir():
    output_dir = output_dir / "02-LLM与Prompt"

result_path = output_dir / "prompt_experiment_3_results.json"
result_path.write_text(
    json.dumps(
        {
            "summary": summary,
            "order_summary": order_summary,
            "records": results,
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("结果已保存：", result_path.resolve())
print("保存记录数：", len(results))

[1/20] c01_old_first - no_priority
[1/20] c01_old_first - priority_rule
[1/20] c01_old_first - latest_only
[2/20] c01_new_first - no_priority
[2/20] c01_new_first - priority_rule
[2/20] c01_new_first - latest_only
[3/20] c02_old_first - no_priority
[3/20] c02_old_first - priority_rule
[3/20] c02_old_first - latest_only
[4/20] c02_new_first - no_priority
[4/20] c02_new_first - priority_rule
[4/20] c02_new_first - latest_only
[5/20] c03_old_first - no_priority
[5/20] c03_old_first - priority_rule
[5/20] c03_old_first - latest_only
[6/20] c03_new_first - no_priority
[6/20] c03_new_first - priority_rule
[6/20] c03_new_first - latest_only
[7/20] c04_old_first - no_priority
[7/20] c04_old_first - priority_rule
[7/20] c04_old_first - latest_only
[8/20] c04_new_first - no_priority
[8/20] c04_new_first - priority_rule
[8/20] c04_new_first - latest_only
[9/20] c05_old_first - no_priority
[9/20] c05_old_first - priority_rule
[9/20] c05_old_first - latest_only
[10/20] c05_new_first - no_priority
[

## 9. 查看顺序影响和失败样本

重点观察：无优先级版本是否更容易选择最后一份文档；有规则和只保留最新版是否消除了顺序影响。

In [18]:
print("按文档顺序统计：")
for item in order_summary:
    print(item)

print("\n失败样本：")
failed_rows = [row for row in results if not row["fully_correct"]]
for row in failed_rows:
    print({
        "id": row["id"],
        "variant": row["variant"],
        "context_source_ids": row["context_source_ids"],
        "expected_source": row["expected"]["source_id"],
        "predicted": row["predicted"],
        "selected_last_document": row["selected_last_document"],
        "error": row["error"],
    })

按文档顺序统计：
{'variant': 'no_priority', 'document_order': 'old_first', 'samples': 10, 'full_accuracy': 1.0, 'selected_last_document_rate': 1.0}
{'variant': 'no_priority', 'document_order': 'new_first', 'samples': 10, 'full_accuracy': 1.0, 'selected_last_document_rate': 0.0}
{'variant': 'priority_rule', 'document_order': 'old_first', 'samples': 10, 'full_accuracy': 1.0, 'selected_last_document_rate': 1.0}
{'variant': 'priority_rule', 'document_order': 'new_first', 'samples': 10, 'full_accuracy': 1.0, 'selected_last_document_rate': 0.0}
{'variant': 'latest_only', 'document_order': 'old_first', 'samples': 10, 'full_accuracy': 1.0, 'selected_last_document_rate': 1.0}
{'variant': 'latest_only', 'document_order': 'new_first', 'samples': 10, 'full_accuracy': 1.0, 'selected_last_document_rate': 1.0}

失败样本：


## 10. 实验结论

### 假设

冲突上下文可能使模型选择旧版口径或受到文档顺序影响。显式优先级规则和程序层过滤最新版都可能提高可靠性，其中检索过滤预计还能减少上下文成本。

### 固定条件（模型、参数、数据集）

- 模型：`glm-4.5-air`
- 参数：关闭 thinking，`temperature=0.1`，`max_tokens=192`
- 输出：三个版本均使用相同 JSON Schema 和 `json_object` 输出模式
- 数据：10 组冲突指标文档，每组交换两次新旧文档顺序，共 20 条样本

### 三种方案

- `no_priority`：新旧文档都进入上下文，不增加优先级规则。
- `priority_rule`：新旧文档都进入上下文，按生效日期和版本选择。
- `latest_only`：程序层过滤旧文档，只把最新版交给模型。

### 结果

| 方案 | API 成功率 | 格式通过率 | 完全正确率 | 来源可追溯率 | 旧版选择率 | 平均耗时 | 平均 Prompt tokens | 总 tokens |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| `no_priority` | 100% | 100% | 100% | 100% | 0% | 1.057 秒 | 469.7 | 10812 |
| `priority_rule` | 100% | 100% | 100% | 100% | 0% | 1.119 秒 | 513.7 | 11722 |
| `latest_only` | 100% | 100% | 100% | 100% | 0% | 1.084 秒 | 391.6 | 9246 |

### 顺序影响和失败样本

60 次请求全部成功，没有格式失败、分类失败或来源编造。对于同时包含新旧文档的方案，最新版在后时模型选择最后一份，最新版在前时仍选择第一份，因此没有观察到机械选择最后一份文档的位置偏差。`latest_only` 的最后文档选择率为 100% 没有比较意义，因为上下文中始终只有一份文档。

显式优先级规则没有提高准确率，却比无优先级方案多消耗约 8.4% 的总 token。`latest_only` 保持相同准确率，总 token 比无优先级方案少约 14.5%，比优先级规则少约 21.1%。三个方案的平均耗时接近，不能据此判断哪种方案稳定更快。

### 实验限制

输入文档包含 `is_latest` 标识，部分模型理由也明确使用了该标识。因此，`no_priority` 虽然没有 Prompt 优先级规则，却已经获得了直接的最新版提示，基线并不真正含糊。实验结果不能证明优先级规则在所有冲突场景中都没有价值，只能说明元数据完整且一致时，模型可以正确使用这些信息。

### 结论和下一步

如果检索层能够可靠识别最新版，应在程序层提前过滤旧文档。这样既能减少 token 成本，也能缩小模型面对的冲突面。Prompt 优先级规则可以作为元数据无法完全治理时的补充，但不应代替版本管理和检索过滤。

若继续实验，应从模型可见的文档中移除 `is_latest`，只在程序内部使用该字段过滤，并加入日期缺失、版本号与日期冲突、错误标记等更困难的样本。